# Week 6 · Objects and lifecycles

An offer is more than data: it has a journey, from new to delivered. This week you read a real class, create objects, and write the rule that decides which moves are allowed.

| | |
|---|---|
| **This week's question** | How does a system keep each thing in a sensible state? |
| **Python you will learn** | reading a class, creating objects, attributes, methods, `self`; a dictionary of allowed moves (a `dict` of lists, `in`) |
| **Principle 6** | **Make wrong states hard to reach.** Allow only the moves that make sense; keep the rule next to the data. |
| **Part you build** | `can_move(old_status, new_status)`: the rule that every status change must pass |

*Campus Food Rescue is fictional. All organisations, people and numbers are invented.*

![System map, week 6: allow only sensible status changes](../assets/diagrams/system_map_w06.svg)

## Before you start

1. **File → Save a copy in Drive**, then work in your copy.
2. Run the setup cell. Run it again after a runtime restart.
3. ⭐ = core for everyone, ⭐⭐ = optional stretch.

In [ ]:
# ▶ Run this cell first (setup)
# In Colab or Jupyter, this cell unpacks the Food Rescue kit next to the notebook.
# This web page is for reading. To run the notebook, use 'Open in Colab' on the course home.


## Class warm-up · Research 4 exchange (10 min)

In groups of three, share the system you researched.

1. What does one record keep, and which **ID** does it use?
2. When something is **corrected**, is the old value kept, replaced, or shown as a correction? How do you know?
3. Who needs the **history**, and what would go wrong without it?

📝 **One idea from a classmate that I want to remember:**

## Session 1 · Learn: objects and their lifecycle (30 min)

### The idea: entities, status and workflow

Think of a parcel you order online. The shop's system follows it: *ordered → shipped → out for delivery → delivered*. The parcel is an **entity**: one real thing that the system keeps track of. Its **status** says where it is on its journey. The **workflow** is the set of moves that make sense. A parcel cannot be delivered before it is shipped, and a delivered parcel does not become "ordered" again.

An offer has the same kind of journey:

![Offer lifecycle: new, approved, picked up, delivered, or rejected or cancelled](../assets/diagrams/lifecycle.svg)

A coordinator **approves** or **rejects** a new offer. Volunteers **pick up** and **deliver** approved food. Every arrow is an allowed move. Every other move is refused. Rejected, cancelled and delivered are **end states**: nothing moves out of them.

### 1 · A class is a blueprint; an object is one thing

Until now, each offer was a dictionary. A **class** goes one step further: it keeps the data **and** the actions that belong to it together. The class is the blueprint. Each **object** made from it is one real thing.

In [ ]:
class Parcel:
    def __init__(self, code, address):
        self.code = code
        self.address = address
        self.status = "ordered"

    def describe(self):
        return f"{self.code} to {self.address}: {self.status}"

parcel_a = Parcel("P01", "Battambang")
parcel_b = Parcel("P02", "Siem Reap")
print(parcel_a.describe())
print(parcel_b.describe())

- `class Parcel:` starts the blueprint. Class names start with a capital letter.
- `__init__` (two underscores on each side) runs automatically when you create an object with `Parcel("P01", "Battambang")`. It sets the starting values.
- `self` means **this object**. `self.status = "ordered"` gives this parcel its own status.
- `describe` is a **method**: a function that belongs to the class. You call it with a dot: `parcel_a.describe()`. Python fills in `self` for you.

### 2 · Attributes: each object keeps its own data

An **attribute** is a value stored on an object, such as `parcel_a.status`. You read it with a dot.

🤔 **Predict** the last line before you run: does changing `parcel_a` also change `parcel_b`?

In [ ]:
print(parcel_a.code, parcel_a.status)
parcel_a.status = "shipped"
print(parcel_a.describe())
print(parcel_b.describe())

**Prediction:** …  **Result:** …

### 3 · Read a real class: `fr.Offer`

In this course you will mostly **read** classes, not write big ones. Reading code is the skill you need to judge what an AI assistant gives you. Here is the source of the kit's own `Offer` class. (You can skip `to_record`; it belongs to Week 7.)

In [ ]:
import inspect
print(inspect.getsource(fr.Offer))

🤔 Read the class and answer:

1. Which attributes does every offer have?
2. What is the status of a brand-new offer? Who decides that?
3. What does `move_to` do **before** it changes the status?

📝 **My answers:**

<details><summary>Show a possible answer</summary>

1. `id`, `donor`, `contact`, `food`, `category`, `portions`, `hours_left`, `status` and `partner`.
2. Always `"new"`, with no partner yet. `__init__` decides it, not the person who creates the offer, so every offer starts at the beginning of its journey.
3. It asks a rule: `is_allowed(self.status, new_status)`. If the answer is `False`, it returns `False` and changes nothing. Only an allowed move changes the status.

</details>

### 4 · Create an offer and call its methods

`fr.Offer(...)` runs `__init__` with your values. Then you can read its attributes and call its methods.

In [ ]:
offer = fr.Offer("F101", "Sunrise Bakery", "Bread rolls", "bread", 30, 5)
print(offer.describe())
print(offer.status, offer.is_open())

### 5 · Python does not stop you

Any code can change any attribute directly. Watch what happens with one typo.

In [ ]:
offer.status = "deliverd"        # a typo, and it skipped every step
print(offer.describe())
print("open?", offer.is_open())

`"deliverd"` is not a real status. The offer is not open, so it disappears from the coordinator's list. It is not `"delivered"` either, so it is not counted as delivered. The bread is simply lost from every report.

Be honest about this: **Python cannot make wrong states impossible.** What a good design does is make them **hard to reach**. The team agrees that the status changes **only** through `move_to`, and `move_to` asks a rule first.

### 6 · A dictionary of allowed moves

A workflow can be stored as data: a dictionary where each key is a status and each value is a **list** of the statuses you may move to. Then `in` answers "is this move allowed?". Start with a traffic light.

🤔 **Predict** each line, then run.

In [ ]:
NEXT_LIGHT = {"green": ["yellow"], "yellow": ["red"], "red": ["green"]}
print(NEXT_LIGHT["green"])
print("yellow" in NEXT_LIGHT["green"])
print("red" in NEXT_LIGHT["green"])
print(fr.ALLOWED_MOVES)

**Predictions:** …  **Results:** …

The last line is the lifecycle picture written as a dictionary. The rule is **data** now: to change the workflow, you change the dictionary, not the logic.

### 7 · `move_to` asks the rule first

`move_to` takes the new status **and** a rule (a function). You have passed functions as values before: `fr.check("urgency", urgency, ...)` and hooks. The kit's own rule is `fr.models.can_move`.

In [ ]:
curry = fr.Offer("F102", "Lotus Cafe", "Vegetable curry", "rice meals", 12, 4)
print(curry.move_to("delivered", fr.models.can_move), curry.status)
print(curry.move_to("approved", fr.models.can_move), curry.status)

🤔 Why does `move_to` return `True` or `False` instead of printing a message?

**My answer:** …

The first move is refused and **nothing changes**; the second is allowed. The rule sits right next to the data it protects: the offer itself refuses a wrong move. That is Principle 6.

## Session 2 · Practise and judge (60 min)

### ⭐ P1 · Read a class and predict

A bus company uses this class. Read it, write your predictions for the four `print` lines, then run.

In [ ]:
class BusTicket:
    def __init__(self, passenger, price_riel):
        self.passenger = passenger
        self.price_riel = price_riel
        self.used = False

    def use(self):
        """A ticket can be used only once. Return True if this use is allowed."""
        if self.used:
            return False
        self.used = True
        return True

ticket = BusTicket("Sokha", 2000)
other = BusTicket("Dara", 1500)
print(ticket.passenger, ticket.used)
print(ticket.use())
print(ticket.use())
print(ticket.used, other.used)

**Predictions:** …  **Results:** …

<details><summary>Show a possible answer</summary>

`Sokha False`, `True`, `False`, `True False`. The second `use()` is refused because the method checks first. `other` is a different object with its own `used`, so it is still `False`. The method protects a rule ("use once"), just as `move_to` protects the lifecycle.

</details>

### ⭐ P2 · Your own class: `PickupTask`

When an offer is approved, a volunteer gets a pickup task. Write a class `PickupTask` with:

- `__init__(self, offer_id, volunteer, portions)` that stores the three values as attributes;
- `describe(self)` that returns a line such as `F001: Sokha collects 30 portions`.

In [ ]:
class PickupTask:
    def __init__(self, offer_id, volunteer, portions):
        # TODO 1: store the three inputs, for example self.offer_id = offer_id
        pass

    def describe(self):
        # TODO 2: return a line like "F001: Sokha collects 30 portions"
        return "?"

task = PickupTask("F001", "Sokha", 30)
line = task.describe()
print(line)
checks = {
    "has the offer ID": "F001" in line,
    "has the volunteer": "Sokha" in line,
    "portions with the unit": "30 portions" in line,
}
for promise, kept in checks.items():
    print("✅" if kept else "❌", promise)
print(PickupTask("F004", "Dara", 12).describe())

<details><summary>Show a possible answer</summary>

```python
class PickupTask:
    def __init__(self, offer_id, volunteer, portions):
        self.offer_id = offer_id
        self.volunteer = volunteer
        self.portions = portions

    def describe(self):
        return f"{self.offer_id}: {self.volunteer} collects {self.portions} portions"
```

Inside a method, always use `self.` to reach the object's own data. Writing only `volunteer` inside `describe` gives a `NameError`, because `describe` has no input with that name.

</details>

### ⭐ P3 · Predict the moves

Use the kit's lifecycle dictionary. Predict each line, then run.

In [ ]:
moves = fr.ALLOWED_MOVES
print(moves["approved"])
print("picked_up" in moves["approved"])
print("delivered" in moves["approved"])
print(moves["delivered"])
print("new" in moves["delivered"])

**Predictions:** …  **Results:** …

<details><summary>Show a possible answer</summary>

`['picked_up', 'cancelled']`, `True`, `False`, `[]`, `False`. An approved offer must be picked up before it can be delivered. A delivered offer has an empty list: no move out at all.

</details>

### ⭐ P4 · Find the end states

Loop over the dictionary and print every status that has **no** moves out (an empty list).

In [ ]:
for status in fr.ALLOWED_MOVES:
    next_statuses = fr.ALLOWED_MOVES[status]
    # TODO: print status only when next_statuses is an empty list

🤔 Why is it important that `delivered` is an end state? What could go wrong if it had a move back to `new`?

**My answer:** …

<details><summary>Show a possible answer</summary>

```python
for status in fr.ALLOWED_MOVES:
    if fr.ALLOWED_MOVES[status] == []:
        print(status, "is an end state")
```

Expected: `delivered`, `rejected`, `cancelled`. If delivered food could go back to new, it could be approved and "delivered" again. A volunteer would travel for food that is already gone, and the same portions would be counted twice.

</details>

### ⭐⭐ S1 · Give `PickupTask` a lifecycle

A task moves `waiting → on_the_way → done`. Copy your class, add `self.status = "waiting"` in `__init__`, and add a method `move_to(self, new_status)` that changes the status only when `TASK_MOVES` allows it. It returns `True` or `False`.

In [ ]:
TASK_MOVES = {"waiting": ["on_the_way"], "on_the_way": ["done"], "done": []}

# Your class here. Then test: "done" straight from "waiting" must be refused.

### ⚖️ Good or risky?

You asked an AI assistant: *"Volunteers need buttons to update an offer, and the coordinator wants a way to fix mistakes."* It wrote **Version A**. **Version B** is a rewrite. Both run without errors.

**Version A**

```python
def update_status(offer, typed_status):
    offer.status = typed_status.strip().lower()   # whatever the volunteer types

def mark_delivered(offer):
    offer.status = "delivered"                    # one click, from any status

def fix_mistake(offer):
    offer.status = "new"                          # reset: start again
```

**Version B**

```python
def update_status(offer, new_status, is_allowed):
    old_status = offer.status
    if offer.move_to(new_status, is_allowed):
        return "done: " + offer.id + " " + old_status + " -> " + new_status
    return "refused: " + offer.id + " cannot move from " + old_status + " to " + new_status
```

Try Version A on one offer and read what happens.

In [ ]:
def update_status(offer, typed_status):
    offer.status = typed_status.strip().lower()

def mark_delivered(offer):
    offer.status = "delivered"

def fix_mistake(offer):
    offer.status = "new"

curry = fr.Offer("F104", "Lotus Cafe", "Vegetable curry with rice", "rice meals", 12, 4)
mark_delivered(curry)
print("1.", curry.describe())
fix_mistake(curry)
print("2.", curry.describe())
update_status(curry, "Deliverd ")
print("3.", curry.describe(), "| open?", curry.is_open())

Use **Principle 6 (make wrong states hard to reach)**. Find **at least three** problems in Version A. For each one, name the offer or person who is harmed.

📝 **Problems I found in Version A:**

1.
2.
3.

<details><summary>Instructor's view</summary>

1. **It skips steps.** `mark_delivered` works from any status. The curry from Lotus Cafe is "delivered" without approval or pickup. No coordinator chose a partner and no volunteer went. Lotus Cafe is told that the curry was collected, so nobody comes for it. The curry spoils at the cafe, no partner receives it, and the delivered total is false.
2. **It resets.** `fix_mistake` sends delivered food back to `new`. The coordinator may approve it again, a volunteer drives to the donor for food that is already gone, and the same portions can be counted as delivered twice. What really happened is lost.
3. **Free text becomes a status.** `"Deliverd "` becomes `"deliverd"`, a status no rule knows. The offer is not open and not delivered: it vanishes from the coordinator's list and from every total except "offered".
4. **No answer.** Version A returns nothing, so the screen cannot tell the volunteer whether the click worked.
5. **No single rule.** Three functions change the status in three different ways. There is no one place to read, test or change the workflow.

Version B has one door (`move_to`), one rule (`is_allowed`) and a clear answer every time. Mistakes are not fixed by a reset. They become an allowed move, such as `approved → cancelled`, and next week the history records why. Version B still cannot stop someone from writing `offer.status = ...` somewhere else. Python cannot make that impossible, so the team rule is "status changes only through `move_to`", and code review checks for it (Week 10).

</details>

## Session 3 · Build your part: the lifecycle rule (60–80 min)

Every time a coordinator or a volunteer presses a button (approve, picked up, delivered), the system calls `move`, and `move` calls `offer.move_to(new_status, can_move)`. **Your** `can_move` decides yes or no. People depend on it:

- the **coordinator**, so that approval cannot be skipped;
- the **volunteers**, so that nobody is sent for food that is already gone;
- the **partners**, so that food arrives only after a real pickup;
- the **reports**, so that every number is honest.

### Your lifecycle and your rule

Build the dictionary from the picture, then write the rule. Keep `REQUIRE_APPROVAL = True` for now; you decide at the end.

In [ ]:
REQUIRE_APPROVAL = True      # your decision (see "Your decision" below)

ALLOWED_MOVES = {
    "new": ["approved", "rejected"],
    "approved": ["picked_up", "cancelled"],
    # TODO 1: add "picked_up", "delivered", "rejected" and "cancelled".
    #         Use the lifecycle picture. An end state has an empty list: [].
}

# TODO 2: if REQUIRE_APPROVAL is False, a new offer may also go straight to
#         "picked_up". Use if, then ALLOWED_MOVES["new"].append("picked_up").


def can_move(old_status, new_status):
    """Return True when the lifecycle allows old_status -> new_status."""
    # TODO 3: if old_status is not a key of ALLOWED_MOVES, return False
    # TODO 4: otherwise return True only when new_status is in its list
    return False

print(can_move("new", "approved"), can_move("new", "delivered"), can_move("deliverd", "new"))

Expected when you are finished: `True False False`. The last call uses an unknown status. Your rule must refuse it, not crash with a `KeyError`.

### Check it

The checker asks eight questions about moves. One of them depends on **your setting**: `can_move("new", "picked_up")` must be `False` when `REQUIRE_APPROVAL` is `True`, and `True` when it is `False`.

In [ ]:
MY_SETTINGS = {"REQUIRE_APPROVAL": REQUIRE_APPROVAL}
fr.check("can_move", can_move, MY_SETTINGS)

This is **one decision, kept consistent in two places**: the setting (`MY_SETTINGS`) and your dictionary. If they disagree, the system says one thing and does another. TODO 2 builds the dictionary **from** the setting, so they can never disagree.

🤔 The starter returned `False` for everything and still passed five cases. Why is a rule that refuses everything "safe" but useless?

**My answer:** …

### Plug it in

Now every move in the real system must pass **your** rule. First the sample day:

In [ ]:
my_system = fr.System(MY_SETTINGS, hooks={"can_move": can_move})
my_system.run_day()

Now try some moves yourself, including three mistakes: delivering before pickup, resetting delivered food, and undoing a rejection.

In [ ]:
attempts = [
    ("F004", "delivered", ""),                        # deliver before pickup
    ("F004", "picked_up", ""),
    ("F004", "delivered", ""),
    ("F002", "new", ""),                              # reset delivered food
    ("F003", "approved", "Hope Children's Centre"),   # undo a rejection
]
for offer_id, new_status, partner in attempts:
    ok, message = my_system.move(offer_id, new_status, partner=partner)
    if ok:
        print("✅", message)
    else:
        print("🚫", message)

When you are finished you see 🚫 ✅ ✅ 🚫 🚫. Each 🚫 is a wrong state that your rule kept out, and the record did not change.

Your rule protects every move that goes through `move`. It cannot stop a line such as `my_system.get("F003").status = "approved"`. That is why the team rule is: **change status only through `move_to`**. Next week, the history will show every move, so a change from nowhere stands out.

### 🧭 Your decision · must a coordinator approve before pickup?

The coordinator approves every offer before pickup. She checks that a partner can take the food and chooses the partner. That protects the partners, but it takes time. Fried rice with 2 hours left might spoil while it waits. Some teams let volunteers collect urgent food first and choose the partner on the way.

- `REQUIRE_APPROVAL = True` (the default): every offer waits for a coordinator.
- `REQUIRE_APPROVAL = False`: a new offer can go straight to picked up. The system prints a ⚠️ **risky setting** warning, and your decision log must justify it.

1. Change `REQUIRE_APPROVAL` in **Your lifecycle and your rule** above, or keep `True` if you can explain why.
2. Run again from that cell down to here. The check still passes, because your dictionary follows the setting.
3. Run the cell below: an urgent offer arrives, and a volunteer wants to collect it **now**.

In [ ]:
offer_id, errors = my_system.add_offer({"donor": "Campus Canteen", "contact": "canteen@example.org",
                                        "food": "Fried rice boxes", "category": "rice meals",
                                        "portions": "20", "hours_left": "2"})
ok, message = my_system.move(offer_id, "picked_up", partner="Hope Children's Centre")
if ok:
    print("✅", message)
else:
    print("🚫", message)

📝 **Decision log · Week 6**

| | |
|---|---|
| **My choice** | `REQUIRE_APPROVAL = …` |
| **An alternative I considered** | |
| **Why I chose it** | |
| **Who is affected (the tradeoff)** | |

### ⭐⭐ Stretch · show only the buttons that make sense

The best place to stop a wrong move is the screen: if the button is not there, nobody can press it. Write `next_moves(status)` that returns the list of statuses an offer can move to, so a volunteer's phone shows only those buttons. An unknown status gets an empty list.

In [ ]:
def next_moves(status):
    return []

for status in ["new", "approved", "picked_up", "delivered", "deliverd"]:
    print(status, "→ buttons:", next_moves(status))

## Finish

- [ ] I shared my Research 4 example in the warm-up.
- [ ] I can read `fr.Offer` and say what `__init__`, `self` and `move_to` do.
- [ ] My `PickupTask` class passes its three checks (P2).
- [ ] I found at least three problems in Version A and named who is harmed.
- [ ] `can_move` passes every check, and my system refuses wrong moves with 🚫.
- [ ] My decision log explains `REQUIRE_APPROVAL`.

### Research 5 · Keeping data safe (5%)

Find out how a real system protects its data from loss and mistakes. Look for **backups**, **audit logs** and **transactions**. Example: a bank transfer makes two updates. Money leaves one account and arrives in another. Both updates must succeed, or neither.

1. Choose a system (mobile money, a bank, school registration, a shop's stock). Name **two updates that must happen together**.
2. What could go wrong **between** the two updates (a power cut, a lost connection, a crash)? Who is harmed?
3. How would you **check the records before and after**, to show that nothing was lost or counted twice?
4. Does the system keep a backup or an audit log? Who is allowed to read it?

Use at least one source and name it. If you use an AI assistant, show one place where you checked or corrected it. Bring your example to the **Week 7 class warm-up**. Submit by email to submit.progmis@gmail.com; the deadline and file name will be announced in class.

**Next week:** right now everything disappears when the notebook closes. You will save records to a file safely and keep a history of every move.